# COLETA, INGESTÃO E PROCESSAMENTO BÁSICO DE DADOS DE UMA PLATAFORMA IOT


### 1. AQUISIÇÃO E INGESTÃO DE DADOS NA CAMADA BRONZE

O script em linguagem Python é executado localmente e permite a coleta de dados de uma aplicação de monitoramento ambiental que estão hospedados na plataforma ThingSpeak.

Ainda localmente, os dados são salvos em formato JSON para posteriomente serem armazenados no bucket criado no Minio.

O programa principal está presente no arquivo **main.py** presente dentro do diretório dataingestion. Sua execução considera parâmetros importantes seguindo a sintaxe:

* `main.py <ip_minio_sever> <api_port_minio_service> <data_source> <temp_dir> <bucket_name> <schedule_time> -o`

Onde:

* **ip_minio_sever:** Endereço IP da máquina remota em que o Minio está em execução

* **api_port_minio_service:** Porta de acesso ao serviço do Minio

* **data_source:** Endereço da fonte de dados

* **temp_dir:** Diretório temporário para armazenar os arquivos json gerados após a coleta dos dados

* **bucket_name:** Nome do bucket de interesse no Minio

* **schedule_time:** Intervalo de tempo (segundos) para a coleta periódica dos dados da plataforma ThingSpeak. (Parâmetro ajustável de acordo com a sua necessidade)

Além disso, tem-se dois parâmetros opcionais:

* **-o:** Ativa exibição de mensagem de sucesso do envio do arquivo para o bucket

O script deve ser executado em um terminal para que suas tarefas sejam executadas enquanto se manipula as instruções neste notebook. Abaixo segue o exemplo de execução do script no meu ambiente:

~~~
python3 main.py 192.168.0.36 9000 https://thingspeak.com/channels/1052510/feed/last.json raw_data/ bronze 300 -o
~~~

Onde:

* **ip_minio_sever:** 192.168.0.36

* **api_port_minio_service:** 9000

* **data_source:** https://thingspeak.com/channels/1052510/feed/last.json

* **temp_dir:** raw_data/

* **bucket_name:** bronze

* **schedule_time:** 300

### 2. PROCESSAMENTO EM LOTE

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.errors import PySparkException
from pyarrow import parquet, fs
from dotenv import load_dotenv
import pyarrow as pa
import sys
import os

load_dotenv()

ENDPOINT = "http://192.168.0.36:9000"
#Keys present in .env file and created by MinIO Platform
ACCESS_KEY = os.getenv("ACCESS_KEY")
SECRET_KEY = os.getenv("SECRET_KEY")

Criando uma sessão Spark e uma conexão remota com o serviço do Minio para acesso aos buckets.

In [2]:
def createSession():
    spark = (
        SparkSession
        .builder
        .master("local[*]")
        .appName("DataSensorPipeline")
        .config("spark.sql.execution.arrow.pyspark.enabled", "true")
        .config("spark.hadoop.fs.s3a.endpoint", ENDPOINT)   
        .config("spark.hadoop.fs.s3a.access.key", ACCESS_KEY)
        .config("spark.hadoop.fs.s3a.secret.key", SECRET_KEY)
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")    
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.path.style.access", True)
        .config("spark.hadoop.fs.s3a.attemps.maximum","1")        
        .getOrCreate()    
    )
    return spark

Criando uma conexão com o serviço do Minio para permitir envio dos dados em formato parquet.

In [3]:
def createMinioConn():
    minio_conn = fs.S3FileSystem(
        endpoint_override=ENDPOINT,
        access_key=ACCESS_KEY,
        secret_key=SECRET_KEY
    )
    return minio_conn

Carga dos objetos presentes em um bucket. 

In [4]:
def readBucket(spark_session, bucket, f_format):
    data_bucket = None
    try:
        #Reading files to DataFrame
        data_bucket = (
                    spark_session             
                        .read
                        .format(f_format)
                        .load(f"s3a://{bucket}/")
            )
    except (PySparkException, TypeError) as e:
        print("Error: ", e)      
        sys.exit(0)
    return data_bucket

##### CAMADA SILVER

Carga dos dados presentes na camada bronze, transformação e submissão para a camada silver

In [ ]:
#Source and destination buckets
src_bucket = "bronze"
dst_bucket = "silver"
format = "json"

#Create a spark session with connection to Minio server
spark_session = createSession()

#Read object from bronze layer
rw_data = readBucket(spark_session, src_bucket, format)

#Change local timezone of the timestamp object
df = rw_data.withColumn("created_at",from_utc_timestamp("created_at","America/Belem"))

df.dtypes

#Split and rename column created_at in year, month, day and time
df_transf = (
            df
            .select(                     
                    year(df["created_at"]).alias('Year'), 
                    date_format(df["created_at"],"MMM").alias('Month'),
                    dayofmonth(df["created_at"]).alias('Day'), 
                    date_format(df["created_at"], 'H:m:s').alias('Time'),
                    df["field3"].cast("float").alias('Humidity'),
                    df["field4"].cast("float").alias("Temperature"),
                    df["field6"].cast("float").alias("Pressure")
                )
            )

#Define schema to pyarrow format
table_schema = pa.schema([    
    ('Year', pa.int16()),
    ('Month', pa.string()),
    ('Day', pa.int16()),
    ('Time', pa.string()),
    ('Humidity', pa.float32()),
    ('Temperature', pa.float32()),    
    ('Pressure', pa.float32()),
])  

#Transform dataframe to table pyarrow format
df_table = pa.Table.from_pandas(df_transf.toPandas(), schema=table_schema)

# Create MinIO connection to send parquet file to the bucket
minio_conn = createMinioConn()

# Convert dataframe to parquet format and write it back to MinIO silver bucket partitioned by year and month
parquet.write_to_dataset(
        table=df_table,
        partition_cols=['Year','Month'],
        root_path=f"{dst_bucket}/",
        filesystem=minio_conn
      )  

spark_session.stop()

Carga dos dados presentes na camada silver.

In [5]:
src_bucket = "silver"
format = 'parquet'

spark_session = createSession()

sv_data = readBucket(spark_session, src_bucket, format)
display(sv_data.toPandas())

spark_session.stop()

23/10/28 14:23:36 WARN Utils: Your hostname, rodrigo resolves to a loopback address: 127.0.1.1; using 192.168.0.11 instead (on interface wlp3s0)
23/10/28 14:23:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
23/10/28 14:23:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
23/10/28 14:23:40 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


,Day,Time,Humidity,Temperature,Pressure,Year,Month
0,28,11:10:57,67.0,36.299999,1008.700012,2023,Oct
1,28,11:10:56,66.0,36.400002,1008.500000,2023,Oct
2,28,11:10:56,67.0,36.299999,1008.599976,2023,Oct
